# Analysis of Results

NOT TO BE EDITED BY AGENTS

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

## Visualize raw data

In [3]:
# Covid 19 US-NY cases & deaths
data = pd.read_csv("data/training/covid19_cases_deaths_us-ny.csv")

In [ ]:
data.date = pd.to_datetime(data.date)

In [5]:
data.head()

,date,case_cnt,death_cnt
0,2020-01-22,0,0
1,2020-01-23,0,0
2,2020-01-24,0,0
3,2020-01-25,0,0
4,2020-01-26,0,0


In [19]:
data['active_cases'] = data['case_cnt'] - data['case_cnt'].shift(15)
data['active_cases'].fillna(0, inplace=True)

/var/folders/9v/_4c_8bdj3x7g3plwm_sjtm3m0000gn/T/ipykernel_43015/2351681227.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['active_cases'].fillna(0, inplace=True)


0           0.0
1           0.0
2           0.0
3           0.0
4           0.0
         ...   
1138    30049.0
1139    32444.0
1140    33470.0
1141    22819.0
1142    24187.0
Name: active_cases, Length: 1143, dtype: float64

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["case_cnt"],
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Cases Over Time — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [21]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["active_cases"],
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Active Cases — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["death_cnt"],
    mode="lines",
    line=dict(width=2, color="#e3a445"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths: %{y:,}<extra></extra>",
))

fig.update_layout(
    title="COVID-19 Deaths Over Time — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Deaths",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

## Set Training/Testing Data and Constants

We'll only consider the first wave of the pandemic for this experiment.

Heuristic: Select start date (first date at which cases > 100) and training cut-off (start date + 60 days)

In [11]:
from datetime import timedelta

In [10]:
start_date = data[data.case_cnt > 100].iloc[0]['date']
start_date

Timestamp('2020-03-08 00:00:00')

In [14]:
train_test_split = start_date + timedelta(days=60)
train_test_split

Timestamp('2020-05-07 00:00:00')

In [16]:
end_date = start_date + timedelta(days=90)
end_date

Timestamp('2020-06-06 00:00:00')

In [23]:
data = data[(data.date >= start_date) & (data.date <= end_date)].copy()

In [24]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["active_cases"],
    mode="lines",
    line=dict(width=2, color="#2a78d6"),
    hovertemplate="%{x|%b %d, %Y}<br>Cases: %{y:,}<extra></extra>",
))

fig.add_vline(
    x=train_test_split,
    line_width=2,
    line_dash="dash",
    line_color="#d62728",
    annotation_text="Train/Test cutoff",
    annotation_position="top left",
)

fig.update_layout(
    title="COVID-19 Active Cases — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Cases",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()

In [26]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data["date"],
    y=data["death_cnt"],
    mode="lines",
    line=dict(width=2, color="#e3a445"),
    hovertemplate="%{x|%b %d, %Y}<br>Deaths: %{y:,}<extra></extra>",
))

fig.add_vline(
    x=train_test_split,
    line_width=2,
    line_dash="dash",
    line_color="#d62728",
    annotation_text="Train/Test cutoff",
    annotation_position="top left",
)

fig.update_layout(
    title="COVID-19 Deaths Over Time — US-NY",
    xaxis_title="Date",
    yaxis_title="Number of Deaths",
    hovermode="x unified",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7"),
    yaxis=dict(gridcolor="#e1e0d9", zeroline=False),
)

fig.show()